## Step 1 — Load and pivot the juror votes

Input file is in long format with columns `Voting_country`, `Juror`, `Participating_country`, `Rank`, `DoB`.
Helper functions live in `jury_helpers.py`.

In [1]:
import pandas as pd
from jury_helpers import (
    DICT_ISO, ISO_TO_COUNTRY,
    rank_to_exp_score, rank_to_points,
    rank_jury_with_tiebreakers,
)

In [2]:
INPUT_FILE = r'jury_votes.xlsx'

long_df = pd.read_excel(INPUT_FILE)

for col in ['Voting_country', 'Participating_country']:
    unknown = set(long_df[col]) - set(ISO_TO_COUNTRY)
    if unknown:
        raise ValueError(f'Unknown ISO codes in {col}: {sorted(unknown)}')
    long_df[col] = long_df[col].map(ISO_TO_COUNTRY)

long_df.head()

,Voting_country,Juror,Participating_country,Rank,DoB
0,France,Juror 1,United Kingdom,1,1998-05-03
1,France,Juror 2,United Kingdom,2,1976-09-06
2,France,Juror 3,United Kingdom,2,1987-04-22
3,France,Juror 1,Spain,3,1998-05-03
4,France,Juror 2,Spain,3,1976-09-06


In [3]:
participating_order = long_df['Participating_country'].drop_duplicates().tolist()
voting_order        = long_df['Voting_country'].drop_duplicates().tolist()
juror_order         = long_df['Juror'].drop_duplicates().tolist()

ranks_df = (
    long_df
    .pivot(index='Participating_country',
           columns=['Voting_country', 'Juror'],
           values='Rank')
    .reindex(index=participating_order,
             columns=pd.MultiIndex.from_product(
                 [voting_order, juror_order],
                 names=['Voting_country', 'Juror']))
)
ranks_df

Voting_country         France                 United Kingdom                  \
Juror                 Juror 1 Juror 2 Juror 3        Juror 1 Juror 2 Juror 3   
Participating_country                                                          
United Kingdom              1       2       2              0       0       0   
Spain                       3       3       3              2       3       3   
France                      0       0       0              1       1       1   
Germany                     2       1       1              3       2       2   

Voting_country        Germany                   Spain                  \
Juror                 Juror 1 Juror 2 Juror 3 Juror 1 Juror 2 Juror 3   
Participating_country                                                   
United Kingdom              2       3       3       2       1       1   
Spain                       1       2       2       0       0       0   
France                      3       1       1       3       2       2   
Germany                     0       0       0       1       3       3   

Voting_country        Portugal                  
Juror                  Juror 1 Juror 2 Juror 3  
Participating_country                           
United Kingdom               4       4       1  
Spain                        1       2       2  
France                       3       1       3  
Germany                      2       3       4

In [4]:
# DoB lookup: {(voting_country, juror): dob}
juror_dobs = (
    long_df.drop_duplicates(['Voting_country', 'Juror'])
           .set_index(['Voting_country', 'Juror'])['DoB']
)
juror_dobs.head()

Voting_country  Juror  
France          Juror 1   1998-05-03
                Juror 2   1976-09-06
                Juror 3   1987-04-22
United Kingdom  Juror 1   1989-06-08
                Juror 2   1959-10-26
Name: DoB, dtype: datetime64[us]

## Step 2 — Convert ranks to exponential scores

In [5]:
exp_scores_df = ranks_df.map(rank_to_exp_score)
exp_scores_df

Voting_country           France                     United Kingdom            \
Juror                   Juror 1   Juror 2   Juror 3        Juror 1   Juror 2   
Participating_country                                                          
United Kingdom         12.00000   9.92351   9.92351        0.00000   0.00000   
Spain                   8.20634   8.20634   8.20634        9.92351   8.20634   
France                  0.00000   0.00000   0.00000       12.00000  12.00000   
Germany                 9.92351  12.00000  12.00000        8.20634   9.92351   

Voting_country                    Germany                         Spain  \
Juror                   Juror 3   Juror 1   Juror 2   Juror 3   Juror 1   
Participating_country                                                     
United Kingdom          0.00000   9.92351   8.20634   8.20634   9.92351   
Spain                   8.20634  12.00000   9.92351   9.92351   0.00000   
France                 12.00000   8.20634  12.00000  12.00000   8.20634   
Germany                 9.92351   0.00000   0.00000   0.00000  12.00000   

Voting_country                             Portugal                      
Juror                   Juror 2   Juror 3   Juror 1   Juror 2   Juror 3  
Participating_country                                                    
United Kingdom         12.00000  12.00000   6.78631   6.78631  12.00000  
Spain                   0.00000   0.00000  12.00000   9.92351   9.92351  
France                  9.92351   9.92351   8.20634  12.00000   8.20634  
Germany                 8.20634   8.20634   9.92351   8.20634   6.78631

## Step 3 — Sum per jury, rank with tie-breakers, award points

Tie-breaker chain (when two countries share the same sum within a jury):
1. Majority of better individual rankings among that jury's jurors.
2. Vote of the youngest juror (largest DoB).
3. Show of hands — the notebook will prompt for the winning ISO code.

In [6]:
jury_sums = exp_scores_df.T.groupby(level='Voting_country', sort=False).sum().T

jury_sums_for_ranking = jury_sums.copy()
for vc in jury_sums_for_ranking.columns:
    if vc in jury_sums_for_ranking.index:
        jury_sums_for_ranking.loc[vc, vc] = pd.NA

jury_sums

Voting_country,France,United Kingdom,Germany,Spain,Portugal
Participating_country,,,,,
United Kingdom,31.84702,0.00000,26.33619,33.92351,25.57262
Spain,24.61902,26.33619,31.84702,0.00000,31.84702
France,0.00000,36.00000,32.20634,28.05336,28.41268
Germany,33.92351,28.05336,0.00000,28.41268,24.91616


In [7]:
jury_ranks = pd.DataFrame(
    index=jury_sums.index, columns=jury_sums.columns, dtype='Int64'
)

for vc in jury_sums.columns:
    sums = jury_sums_for_ranking[vc]
    ranks_in_jury = ranks_df.xs(vc, axis=1, level='Voting_country')
    dobs_for_jury = juror_dobs.loc[vc].to_dict()
    rank_series = rank_jury_with_tiebreakers(
        vc, sums, ranks_in_jury, dobs_for_jury
    )
    for c, r in rank_series.items():
        jury_ranks.loc[c, vc] = r

jury_ranks

Voting_country,France,United Kingdom,Germany,Spain,Portugal
Participating_country,,,,,
United Kingdom,2,<NA>,3,1,3
Spain,3,3,2,<NA>,1
France,<NA>,1,1,3,2
Germany,1,2,<NA>,2,4


In [8]:
jury_points = jury_ranks.map(rank_to_points).astype(int)
jury_points

Voting_country,France,United Kingdom,Germany,Spain,Portugal
Participating_country,,,,,
United Kingdom,10,0,8,12,8
Spain,8,8,10,0,12
France,0,12,12,8,10
Germany,12,10,0,10,7


## Step 4 — Final classification

In [9]:
final_scores = jury_points.sum(axis=1)
final_ranks  = final_scores.rank(ascending=False, method='min').astype(int)

final_classification = (
    pd.DataFrame({'Total_points': final_scores, 'Rank': final_ranks})
    .sort_values('Rank')
)
final_classification

,Total_points,Rank
Participating_country,,
France,42,1
Germany,39,2
Spain,38,3
United Kingdom,38,3
